# Task 4: RNA, Copy Number & Clinical Status Correlation

**Goal:** Determine whether RNA expression or DNA copy number is more predictive of clinical HER2 status (IHC/FISH).

Steps:
1. Load all processed data and merge
2. Correlate ERBB2 RNA expression with HER2 clinical status
3. Correlate ERBB2 copy number with HER2 clinical status
4. Compare predictive performance (ROC AUC, or boxplots)
5. Optional: logistic regression model

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from src.data_utils import load_processed, load_copy_number
from src.plotting import plot_expression_boxplot, save_fig

## 1. Load and merge data

In [ ]:
rna = load_processed('rna_normalized')
clinical = (
    load_processed('clinical_cleaned')
    .set_index('Patient ID')
    .loc[lambda df: ~df.index.duplicated(keep='first')]
)
cn = load_copy_number()

print('RNA:', rna.shape)
print('Copy Number:', cn.shape)
print('Clinical:', clinical.shape)

In [ ]:
# Build a merged analysis DataFrame
analysis = pd.DataFrame(index=rna.index)
analysis['erbb2_rna'] = rna.get('ERBB2')  # log2 CPM
analysis['her2_positive'] = clinical.reindex(rna.index)['her2_positive']

# Align copy number, coerce to nullable integer (pd.Int8Dtype keeps NaN distinct from 0)
_cn_raw = cn.reindex(rna.index)['erbb2_copy_number']
_cn_valid = pd.to_numeric(_cn_raw, errors='coerce')              # non-numeric → NaN
_cn_valid = _cn_valid.where(_cn_valid.isin([-2, -1, 0, 1, 2]))  # discard out-of-range values
analysis['erbb2_cn'] = _cn_valid.astype(pd.Int8Dtype())          # integer with NA support

analysis = analysis.dropna(subset=['erbb2_rna', 'her2_positive'])
print('Merged samples:', analysis.shape)
print('erbb2_cn dtype:', analysis['erbb2_cn'].dtype)
print('erbb2_cn value counts:\n', analysis['erbb2_cn'].value_counts().sort_index())

## 2. ERBB2 RNA vs HER2 clinical status

In [ ]:
from scipy.stats import mannwhitneyu

analysis['her2_label'] = analysis['her2_positive'].map({True: 'HER2+', False: 'HER2-'})

# Mann-Whitney U test between HER2+ and HER2- ERBB2 expression
her2_pos = analysis.loc[analysis['her2_positive'] == True, 'erbb2_rna']
her2_neg = analysis.loc[analysis['her2_positive'] == False, 'erbb2_rna']
stat, pval = mannwhitneyu(her2_pos, her2_neg, alternative='two-sided')

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=analysis, x='her2_label', y='erbb2_rna', order=['HER2-', 'HER2+'], palette=['#3498DB', '#E74C3C'], ax=ax)

# Annotate p-value bracket between the two boxes
y_max = analysis['erbb2_rna'].max()
y_bracket = y_max * 1.05
bracket_h = y_max * 0.02
ax.plot([0, 0, 1, 1], [y_bracket, y_bracket + bracket_h, y_bracket + bracket_h, y_bracket], color='black', linewidth=1)
p_text = f'p = {pval:.2e}' if pval < 0.001 else f'p = {pval:.3f}'
ax.text(0.5, y_bracket + bracket_h * 1.5, p_text, ha='center', va='bottom', fontsize=10)

ax.set_title('ERBB2 RNA Expression by HER2 Clinical Status')
ax.set_ylabel('ERBB2 (log2 CPM)')
ax.set_xlabel('')
save_fig('04_erbb2_rna_boxplot')
plt.show()

print(f'Mann-Whitney U = {stat:.0f},  {p_text}  (n HER2+={len(her2_pos)}, n HER2-={len(her2_neg)})')

## 3. ERBB2 copy number vs HER2 clinical status

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(
    data=analysis.dropna(subset=['erbb2_cn']),
    x='erbb2_cn',
    hue='her2_label',
    palette={'HER2-': '#3498DB', 'HER2+': '#E74C3C'},
    ax=ax,
)
ax.set_title('ERBB2 Copy Number vs HER2 Clinical Status')
ax.set_xlabel('Copy Number (−2=deep del, 0=diploid, 2=high amp)')
ax.set_ylabel('Samples')
ax.legend(title='HER2 Status')
save_fig('04_erbb2_cn_barplot')
plt.show()

In [ ]:
from scipy.stats import kruskal

cn_order = [-2, -1, 0, 1, 2]
cn_labels = {-2: 'Deep del\n(−2)', -1: 'Shallow del\n(−1)', 0: 'Diploid\n(0)', 1: 'Low amp\n(+1)', 2: 'High amp\n(+2)'}

plot_data = analysis.dropna(subset=['erbb2_cn', 'erbb2_rna']).copy()
plot_data['cn_label'] = plot_data['erbb2_cn'].map(cn_labels)
label_order = [cn_labels[k] for k in cn_order if k in plot_data['erbb2_cn'].unique()]

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=plot_data, x='cn_label', y='erbb2_rna',
    order=label_order,
    palette='RdYlBu_r', ax=ax, fliersize=2,
)
ax.set_title('ERBB2 RNA Expression by Copy Number Category')
ax.set_xlabel('ERBB2 Copy Number')
ax.set_ylabel('ERBB2 (log2 CPM)')

# Kruskal-Wallis test across all CN groups
groups = [plot_data.loc[plot_data['erbb2_cn'] == cn, 'erbb2_rna'].values for cn in cn_order if cn in plot_data['erbb2_cn'].values]
stat, pval = kruskal(*groups)
p_text = f'Kruskal-Wallis p = {pval:.2e}' if pval < 0.001 else f'Kruskal-Wallis p = {pval:.3f}'
ax.text(0.98, 0.97, p_text, transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8))

# Annotate n per group
for i, cn in enumerate(cn_order):
    n = (plot_data['erbb2_cn'] == cn).sum()
    if n > 0:
        ax.text(i, ax.get_ylim()[0], f'n={n}', ha='center', va='bottom', fontsize=8, color='gray')

save_fig('04_erbb2_rna_by_cn')
plt.show()

In [ ]:
cn_order = [-2, -1, 0, 1, 2]
cn_labels = {-2: '−2\n(deep del)', -1: '−1\n(shallow del)', 0: '0\n(diploid)', 1: '+1\n(low amp)', 2: '+2\n(high amp)'}

cn_plot = analysis.dropna(subset=['erbb2_cn', 'her2_label']).copy()
cn_plot['erbb2_cn'] = cn_plot['erbb2_cn'].astype(int)

# Compute % of each copy number category within each HER2 group
counts = (
    cn_plot.groupby(['her2_label', 'erbb2_cn'], observed=True)
    .size()
    .reset_index(name='n')
)
totals = cn_plot.groupby('her2_label', observed=True).size().rename('total')
counts = counts.join(totals, on='her2_label')
counts['pct'] = counts['n'] / counts['total'] * 100

# Pivot for grouped bar chart
pivot = counts.pivot(index='erbb2_cn', columns='her2_label', values='pct').reindex(cn_order).fillna(0)

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(cn_order))
width = 0.35
colors = {'HER2-': '#3498DB', 'HER2+': '#E74C3C'}

for i, (label, color) in enumerate(colors.items()):
    if label in pivot.columns:
        offsets = [xi + (i - 0.5) * width for xi in x]
        bars = ax.bar(offsets, pivot[label], width=width, label=label, color=color, alpha=0.8, edgecolor='white')
        # Annotate bars with % values
        for bar, val in zip(bars, pivot[label]):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                        f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(list(x))
ax.set_xticklabels([cn_labels[c] for c in cn_order], fontsize=9)
ax.set_xlabel('ERBB2 Copy Number Category')
ax.set_ylabel('% of samples within HER2 group')
ax.set_title('ERBB2 Copy Number Distribution by HER2 Clinical Status')
ax.legend(title='HER2 Status')
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
plt.tight_layout()
save_fig('04_erbb2_cn_by_her2')
plt.show()

# Print contingency table
print(pivot.T.to_string(float_format='{:.1f}%'.format))


## 4. Predictive performance: ROC AUC

In [ ]:
y = analysis['her2_positive'].astype(int)

# RNA model
X_rna = analysis[['erbb2_rna']].fillna(0)
auc_rna = roc_auc_score(y, X_rna)
print(f'RNA ERBB2 AUC: {auc_rna:.3f}')

# TODO: CN model when copy number is aligned
# X_cn = analysis[['erbb2_cn']].fillna(0)
# auc_cn = roc_auc_score(y, X_cn)
# print(f'CN ERBB2 AUC: {auc_cn:.3f}')

In [ ]:
# ROC curve for RNA
fpr, tpr, _ = roc_curve(y, X_rna)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'RNA (AUC={auc_rna:.2f})', color='#E74C3C')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC: ERBB2 RNA predicting HER2 Clinical Status')
plt.legend()
save_fig('04_roc_rna')
plt.show()

## 5. Logistic regression (optional)

In [ ]:
# TODO: extend with multi-gene RNA features and/or copy number
lr = LogisticRegression()
cv_scores = cross_val_score(lr, X_rna, y, cv=5, scoring='roc_auc')
print(f'5-fold CV AUC (RNA): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')